#Tahap 2
Data hasil cleaning_01_Industrial_Data_Operations_Profiling

##ETL & Data Quality Pipeline

- Extract: membaca data mentah dari lima CSV.
- Transform: membersihkan tipe data dan memvalidasi kualitas data.
- Load: menyimpan data hasil transformasi ke folder processed.
- Membuat laporan sebelum dan sesudah proses.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


##1.Masukkan dataset mentah ke Google Drive

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/industrial-data-operations"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [RAW_DIR, PROCESSED_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)
print("Reports folder:", REPORT_DIR)

Project folder: /content/drive/MyDrive/industrial-data-operations
Raw data folder: /content/drive/MyDrive/industrial-data-operations/data/raw
Processed data folder: /content/drive/MyDrive/industrial-data-operations/data/processed
Reports folder: /content/drive/MyDrive/industrial-data-operations/reports


Kode ini hanya mengekstrak file CSV ke folder raw. File asli di ZIP tidak diubah.

##2.Extract: baca semua CSV
Sekarang kita baca kelima dataset dengan Pandas

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()

zip_files = [
    filename
    for filename in uploaded
    if filename.lower().endswith(".zip")
]

if len(zip_files) != 1:
    raise ValueError(
        "Upload tepat satu file ZIP dataset."
    )

with zipfile.ZipFile(zip_files[0], "r") as archive:
    csv_members = [
        member for member in archive.namelist()
        if member.lower().endswith(".csv")
        and not member.startswith("__MACOSX/")
    ]

    for member in csv_members:
        output_path = RAW_DIR / Path(member).name

        with archive.open(member) as source:
            output_path.write_bytes(source.read())

print("CSV yang berhasil diekstrak:", len(csv_members))

for path in sorted(RAW_DIR.glob("*.csv")):
    print(path.name, "-", round(path.stat().st_size / 1024, 2), "KB")

Saving Microsoft Azure Predictive Maintenance.zip to Microsoft Azure Predictive Maintenance.zip
CSV yang berhasil diekstrak: 5
PdM_errors.csv - 126.05 KB
PdM_failures.csv - 23.77 KB
PdM_machines.csv - 1.54 KB
PdM_maint.csv - 102.44 KB
PdM_telemetry.csv - 78263.99 KB


##3.Periksa struktur dan buat salinan kerja
Sebelum transformasi, kita periksa bahwa file yang dibaca sesuai dengan dataset yang telah kamu profilkan

In [ ]:
import pandas as pd

# 1. Baca semua dataset dari folder raw ke raw_data
raw_data = {}
for path in RAW_DIR.glob("*.csv"):
    raw_data[path.stem] = pd.read_csv(path)

expected_files = {
    "pdm_errors": "PdM_errors",
    "pdm_failures": "PdM_failures",
    "pdm_machines": "PdM_machines",
    "pdm_maint": "PdM_maint",
    "pdm_telemetry": "PdM_telemetry"
}

# Normalisasi nama file agar tidak sensitif huruf besar/kecil
available_data = {
    name.lower(): df
    for name, df in raw_data.items()
}

missing_files = set(expected_files) - set(available_data)

print("File yang ditemukan:", list(raw_data.keys()))

if missing_files:
    raise ValueError(
        f"Dataset belum lengkap: {sorted(missing_files)}. "
        "Periksa kembali folder data/raw."
    )

# Buat salinan kerja tanpa mengubah data mentah
clean_data = {
    canonical_name: available_data[lower_name].copy(deep=True)
    for lower_name, canonical_name in expected_files.items()
}

print("\nSemua 5 dataset tersedia.")
print("Salinan kerja berhasil dibuat.\n")

for name, df in clean_data.items():
    print(f"{name}: {df.shape}")

File yang ditemukan: ['PdM_errors', 'PdM_failures', 'PdM_machines', 'PdM_maint', 'PdM_telemetry']

Semua 5 dataset tersedia.
Salinan kerja berhasil dibuat.

PdM_errors: (3919, 3)
PdM_failures: (761, 3)
PdM_machines: (100, 3)
PdM_maint: (3286, 3)
PdM_telemetry: (876100, 6)


##4.Validasi data
Sebelum membersihkan data, kita perlu memastikan struktur dan kualitas data masih sesuai dengan hasil profiling Tahap 1.

In [ ]:
validation_report = []

for name, df in clean_data.items():
    validation_report.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "column_names": ", ".join(df.columns)
    })

validation_df = pd.DataFrame(validation_report)

display(validation_df)


,dataset,rows,columns,missing_values,duplicate_rows,column_names
0,PdM_errors,3919,3,0,0,"datetime, machineID, errorID"
1,PdM_failures,761,3,0,0,"datetime, machineID, failure"
2,PdM_machines,100,3,0,0,"machineID, model, age"
3,PdM_maint,3286,3,0,0,"datetime, machineID, comp"
4,PdM_telemetry,876100,6,0,0,"datetime, machineID, volt, rotate, pressure, v..."


##5.Transformasi tipe data
Tujuan tahap ini adalah mengubah kolom datetime menjadi format tanggal dan waktu yang benar, serta memastikan kolom machineID menggunakan tipe data numerik yang sesuai.

In [ ]:
# Simpan jumlah baris sebelum transformasi
rows_before = {
    name: len(df)
    for name, df in clean_data.items()
}

# 1. Ubah kolom datetime menjadi datetime64
for name, df in clean_data.items():
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(
            df["datetime"],
            errors="coerce"
        )

# 2. Pastikan machineID bertipe integer
for name, df in clean_data.items():
    if "machineID" in df.columns:
        machine_id = pd.to_numeric(
            df["machineID"],
            errors="coerce"
        )

        # Hentikan proses jika ada ID yang tidak valid
        if machine_id.isna().any():
            raise ValueError(
                f"Ada machineID tidak valid di {name}"
            )

        # Pastikan semua ID berupa bilangan bulat
        if not machine_id.mod(1).eq(0).all():
            raise ValueError(
                f"Ada machineID bukan bilangan bulat di {name}"
            )

        df["machineID"] = machine_id.astype("int64")

print("Transformasi tipe data selesai.")


Transformasi tipe data selesai.


Periksa hasil transformasi

In [ ]:
# Tahap 2.3 — Validasi hasil transformasi

type_report = []

for name, df in clean_data.items():
    type_report.append({
        "dataset": name,
        "rows_before": rows_before[name],
        "rows_after": len(df),
        "row_count_unchanged": (
            rows_before[name] == len(df)
        ),
        "datetime_dtype": (
            str(df["datetime"].dtype)
            if "datetime" in df.columns
            else "N/A"
        ),
        "machineID_dtype": str(df["machineID"].dtype),
        "missing_values_after": int(
            df.isna().sum().sum()
        )
    })

type_report_df = pd.DataFrame(type_report)

display(type_report_df)


,dataset,rows_before,rows_after,row_count_unchanged,datetime_dtype,machineID_dtype,missing_values_after
0,PdM_errors,3919,3919,True,datetime64[ns],int64,0
1,PdM_failures,761,761,True,datetime64[ns],int64,0
2,PdM_machines,100,100,True,N/A,int64,0
3,PdM_maint,3286,3286,True,datetime64[ns],int64,0
4,PdM_telemetry,876100,876100,True,datetime64[ns],int64,0


Catatan penting: jika konversi tanggal menghasilkan nilai kosong atau NaT, proses berikutnya harus dihentikan dan diperiksa. Jangan langsung menghapus baris tersebut

##6.Simpan data yang sudah ditransformasi

In [ ]:

# Tahap 2.4 — Export processed data

for name, df in clean_data.items():
    output_path = PROCESSED_DIR / f"{name}_processed.csv"

    df.to_csv(
        output_path,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S"
    )

    print(f"Tersimpan: {output_path.name} | {df.shape}")

print("\nSemua dataset berhasil diekspor.")


Tersimpan: PdM_errors_processed.csv | (3919, 3)
Tersimpan: PdM_failures_processed.csv | (761, 3)
Tersimpan: PdM_machines_processed.csv | (100, 3)
Tersimpan: PdM_maint_processed.csv | (3286, 3)
Tersimpan: PdM_telemetry_processed.csv | (876100, 6)

Semua dataset berhasil diekspor.


##7. Buat Laporan Data Quality

Kita akan mencatat kondisi setiap dataset setelah transformasi, termasuk jumlah baris, missing values, duplikasi, dan validitas tanggal.

In [ ]:
quality_report = []

for name, df in clean_data.items():
    datetime_invalid = (
        int(df["datetime"].isna().sum())
        if "datetime" in df.columns
        else 0
    )

    quality_report.append({
        "dataset": name,
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "invalid_datetime": datetime_invalid,
        "machineID_unique": (
            int(df["machineID"].nunique())
            if "machineID" in df.columns
            else None
        ),
        "quality_status": (
            "PASS"
            if (
                df.isna().sum().sum() == 0
                and df.duplicated().sum() == 0
                and datetime_invalid == 0
            )
            else "REVIEW"
        )
    })

quality_df = pd.DataFrame(quality_report)

display(quality_df)

# Simpan laporan ke Google Drive
quality_path = REPORT_DIR / "data_quality_report.csv"
quality_df.to_csv(quality_path, index=False)

print(f"\nLaporan tersimpan: {quality_path}")


,dataset,total_rows,total_columns,missing_values,duplicate_rows,invalid_datetime,machineID_unique,quality_status
0,PdM_errors,3919,3,0,0,0,100,PASS
1,PdM_failures,761,3,0,0,0,98,PASS
2,PdM_machines,100,3,0,0,0,100,PASS
3,PdM_maint,3286,3,0,0,0,100,PASS
4,PdM_telemetry,876100,6,0,0,0,100,PASS



Laporan tersimpan: /content/drive/MyDrive/industrial-data-operations/reports/data_quality_report.csv


##8.Dokumentasikan hasil ETL


In [ ]:

# Tahap 2.7 — ETL Summary

summary = {
    "project": "Industrial Data Operations Pipeline",
    "stage": "Stage 2 - ETL and Data Cleaning",
    "datasets_processed": len(clean_data),
    "total_rows_processed": sum(
        len(df) for df in clean_data.values()
    ),
    "total_missing_values": int(
        quality_df["missing_values"].sum()
    ),
    "total_duplicate_rows": int(
        quality_df["duplicate_rows"].sum()
    ),
    "datasets_passed": int(
        (quality_df["quality_status"] == "PASS").sum()
    ),
    "datasets_need_review": int(
        (quality_df["quality_status"] == "REVIEW").sum()
    )
}

for key, value in summary.items():
    print(f"{key}: {value}")

# Simpan ringkasan
summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["metric", "value"]
)

summary_path = REPORT_DIR / "etl_summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"\nRingkasan tersimpan: {summary_path}")


project: Industrial Data Operations Pipeline
stage: Stage 2 - ETL and Data Cleaning
datasets_processed: 5
total_rows_processed: 884166
total_missing_values: 0
total_duplicate_rows: 0
datasets_passed: 5
datasets_need_review: 0

Ringkasan tersimpan: /content/drive/MyDrive/industrial-data-operations/reports/etl_summary.csv
